<div dir="rtl">

# פרק 7: פונקציות מתמטיות

*המודול math, ניתוח פונקציה ומציאת נקודה שבה היא מגיעה לערך מסוים, ופתרון משוואות עם sympy.*

**איך לעבוד עם המחברת:** הרץ כל תא קוד עם Shift+Enter. בתרגילים, כתוב את הפתרון בתא הריק, ורק אחר כך פתח את הפתרון בלחיצה כפולה על התא שמתחתיו. כדי לשמור את העבודה שלך: **File → Save a copy in GitHub** (או **Save a copy in Drive**).

## המודול math

</div>

In [ ]:
import math

print(math.sqrt(16))          # square root
print(math.pi)
print(math.exp(1))            # e to the power of 1
print(math.log(100, 10))      # log base 10
print(math.log(math.e))       # natural log (ln)
print(math.sin(math.pi / 2))  # angles in radians

<div dir="rtl">

**פלט צפוי:**

```text
4.0
3.141592653589793
2.718281828459045
2.0
1.0
1.0
```

> 💡 **log בפייתון**  
> `math.log(x)` בלי בסיס הוא לוגריתם טבעי (ln). ללוגריתם בבסיס 10 כותבים `math.log10(x)` או `math.log(x, 10)`.

## פונקציה מתמטית כפונקציה של פייתון

כל נוסחה אפשר לכתוב כפונקציה. דוגמה מפרמקולוגיה: פינוי תרופה מהדם. הריכוז יורד מעריכית, `C(t) = C0 · e-kt`, כאשר `k = ln(2) / t½`.

</div>

In [ ]:
def concentration(t, c0=100, half_life=6):
    k = math.log(2) / half_life
    return c0 * math.exp(-k * t)

for t in range(0, 25, 6):
    print(t, round(concentration(t), 2))

<div dir="rtl">

**פלט צפוי:**

```text
0 100.0
6 50.0
12 25.0
18 12.5
24 6.25
```

כל 6 שעות הריכוז יורד בחצי, בדיוק כמו שזמן מחצית חיים אמור לעשות. כדאי לבדוק נוסחה על ערכים שאתה יודע מה התשובה שלהם.

## ציור הפונקציה

כדי לצייר פונקציה, מחשבים אותה בהרבה נקודות קרובות. כאן כל חצי שעה מ-0 עד 30:

</div>

In [ ]:
import matplotlib.pyplot as plt

times = [t / 2 for t in range(0, 61)]       # 0, 0.5, 1, ... 30
conc = [concentration(t) for t in times]

plt.plot(times, conc)
plt.axhline(10, color="gray", linestyle=":")
plt.xlabel("Hours")
plt.ylabel("Concentration (mg/L)")
plt.title("Drug elimination, half-life 6h")
plt.show()

<div dir="rtl">

בפרק 9 נלמד את `numpy.linspace`, שעושה את זה בקלות רבה יותר.

## מתי הריכוז יורד מתחת לסף?

### דרך 1: פתרון אלגברי

מבודדים את t מהנוסחה: `t = ln(C0 / C) / k`.

</div>

In [ ]:
k = math.log(2) / 6
t = math.log(100 / 10) / k
print(f"{t:.2f} hours")

<div dir="rtl">

**פלט צפוי:**

```text
19.93 hours
```

### דרך 2: חיפוש נומרי בשיטת החצייה (bisection)

לא תמיד אפשר לבודד את המשתנה. אז מחפשים את התשובה במספרים: אם הפונקציה מחליפה סימן בין a ל-b, התשובה נמצאת ביניהם. בודקים את האמצע, וממשיכים עם החצי שבו השינוי קורה. אחרי כמה עשרות צעדים מגיעים לדיוק גבוה מאוד.

</div>

In [ ]:
def bisection(f, a, b, tol=1e-6):
    while b - a > tol:
        mid = (a + b) / 2
        if f(a) * f(mid) <= 0:    # sign changes in the left half
            b = mid
        else:
            a = mid
    return (a + b) / 2

def gap(t):
    return concentration(t) - 10    # zero when concentration == 10

print(round(bisection(gap, 0, 100), 3))

<div dir="rtl">

**פלט צפוי:**

```text
19.932
```

שתי הדרכים נותנות אותה תשובה. שים לב שהעברנו **פונקציה** (`gap`) כפרמטר לפונקציה אחרת. בפייתון זה מותר ושימושי מאוד.

> 💡 **lambda**  
> אפשר לכתוב פונקציה קצרה בלי def: `bisection(lambda t: concentration(t) - 10, 0, 100)`. זה שקול לחלוטין להגדרת `gap`.

## sympy: מתמטיקה עם סמלים

עד עכשיו פייתון חישב מספרים. sympy עובד עם **ביטויים**, כמו שעושים על דף: גוזר, מפשט ופותר משוואות.

</div>

In [ ]:
import sympy as sp

x = sp.symbols("x")
print(sp.solve(x**2 - 5*x + 6, x))         # roots of the equation = 0
print(sp.diff(x**3 + 2*x, x))              # derivative
print(sp.simplify((x**2 - 1) / (x - 1)))   # simplify

<div dir="rtl">

**פלט צפוי:**

```text
[2, 3]
3*x**2 + 2
x + 1
```

### ניתוח נוסחת הפינוי

</div>

In [ ]:
t, k, C0 = sp.symbols("t k C0", positive=True)
C = C0 * sp.exp(-k * t)

print(sp.diff(C, t))                          # rate of change
print(sp.solve(sp.Eq(C, C0 / 2), t))          # when is C half of C0?
print(sp.integrate(C, (t, 0, sp.oo)))         # area under the curve (AUC)

<div dir="rtl">

**פלט צפוי:**

```text
-C0*k*exp(-k*t)
[log(2)/k]
C0/k
```

שלוש תוצאות מוכרות מפרמקולוגיה: קצב הירידה פרופורציונלי לריכוז, זמן מחצית החיים הוא `ln(2)/k`, והשטח מתחת לעקומה (AUC) הוא `C0/k`. `sp.oo` הוא אינסוף, ו-`positive=True` אומר ל-sympy שהמשתנים חיוביים, וכך הוא יכול לפשט יותר.

כדי לקבל מספר מביטוי, מציבים ערכים: `C.subs({C0: 100, k: 0.1155, t: 6})`.

## תרגילים

### ✏️ תרגיל 7.1 (קל)

כתוב פונקציה `bsa(weight, height_cm)` שמחשבת שטח פנים של הגוף בנוסחת Mosteller: השורש של (משקל × גובה בס"מ / 3600). בדוק עם 70 ק"ג ו-175 ס"מ.

</div>

In [ ]:
# תרגיל 7.1: כתוב כאן את הפתרון שלך

In [ ]:
#@title 👀 הצג פתרון לתרגיל 7.1 { display-mode: "form" }
# double-click this cell to see the solution

import math
def bsa(weight, height_cm):
    return math.sqrt(weight * height_cm / 3600)

print(round(bsa(70, 175), 2))

# expected output:
# 1.84

<div dir="rtl">

### ✏️ תרגיל 7.2 (בינוני)

לתרופה זמן מחצית חיים של 4 שעות וריכוז התחלתי 80 mg/L. השתמש ב-`bisection` כדי למצוא מתי הריכוז מגיע ל-5 mg/L, ובדוק את התשובה בנוסחה האלגברית.

</div>

In [ ]:
# תרגיל 7.2: כתוב כאן את הפתרון שלך

In [ ]:
#@title 👀 הצג פתרון לתרגיל 7.2 { display-mode: "form" }
# double-click this cell to see the solution

print(round(bisection(lambda t: concentration(t, 80, 4) - 5, 0, 100), 3))

k = math.log(2) / 4
print(round(math.log(80 / 5) / k, 3))

# expected output:
# 16.0
# 16.0
# 80 ← 40 ← 20 ← 10 ← 5: ארבע מחציות חיים, כלומר 16 שעות.

<div dir="rtl">

### ✏️ תרגיל 7.3 (בינוני)

עם sympy: מצא את הנגזרת של `x**2 * sp.exp(-x)`, ואת הנקודות שבהן הנגזרת מתאפסת.

</div>

In [ ]:
# תרגיל 7.3: כתוב כאן את הפתרון שלך

In [ ]:
#@title 👀 הצג פתרון לתרגיל 7.3 { display-mode: "form" }
# double-click this cell to see the solution

x = sp.symbols("x")
f = x**2 * sp.exp(-x)
df = sp.diff(f, x)
print(df)
print(sp.solve(df, x))

# expected output:
# -x**2*exp(-x) + 2*x*exp(-x)
# [0, 2]
# בנקודות האלה הפונקציה מגיעה למינימום או למקסימום (x=0 מינימום, x=2 מקסימום).

<div dir="rtl">

## ההערות שלי

כתוב כאן מה הבנת, מה לא ברור, ומה לשאול בתרגול. לחיצה כפולה על התא פותחת אותו לעריכה.

-

</div>